# Prepare PARROT data for MHC-I binding peptides

Prepare sequences and measured scores for MHC-I binding peptides, split the data into training, validation, and test sets, and write the files used by PARROT.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.unicode_minus'] = False

## Data split

Split the data into training, validation, and test sets at a ratio of 0.8 : 0.1 : 0.1.

## Load and prepare the data

In [2]:
# Load the peptide data.
plm_data = "...\\pLM-bVAE\\reproduction_paper\\data\\"
pep_data = pd.read_csv(plm_data + "raw_data\\MHCIpeptide_bdata.20130222.mhci.txt", sep="\t")

In [3]:
# Select HLA-A*02:01 binding peptides.
pep_data_filtered = pep_data[pep_data['mhc'] == "HLA-A*02:01"].reset_index(drop=True)  


In [4]:
pep_data_filtered 

,species,mhc,peptide_length,sequence,inequality,meas
0,human,HLA-A*02:01,8,ILGFVFTL,=,1.225423
1,human,HLA-A*02:01,8,FLGRIWPS,=,2.000000
2,human,HLA-A*02:01,8,FLGKIWPS,=,4.000000
3,human,HLA-A*02:01,8,FLGKIWSS,=,8.000000
4,human,HLA-A*02:01,8,GLAVAMEV,=,31.808753
...,...,...,...,...,...,...
12155,human,HLA-A*02:01,15,EISTNIRQAGVQYSR,>,178571.428600
12156,human,HLA-A*02:01,15,IRQAGVQYSRADEEQ,>,178571.428600
12157,human,HLA-A*02:01,17,QFLKFSLPFPFLYKFLL,=,20681.845230
12158,human,HLA-A*02:01,18,PLMGGAYIAFPTSCHMFI,=,13161.358710


In [5]:
# Check for duplicate sequences.
print(pep_data_filtered['sequence'].duplicated().sum())

0


In [6]:
# Apply a log transformation to the measured values.
pep_data_filtered['log_meas'] = np.log(pep_data_filtered['meas'])

In [7]:
pep_data_filtered

,species,mhc,peptide_length,sequence,inequality,meas,log_meas
0,human,HLA-A*02:01,8,ILGFVFTL,=,1.225423,0.203286
1,human,HLA-A*02:01,8,FLGRIWPS,=,2.000000,0.693147
2,human,HLA-A*02:01,8,FLGKIWPS,=,4.000000,1.386294
3,human,HLA-A*02:01,8,FLGKIWSS,=,8.000000,2.079442
4,human,HLA-A*02:01,8,GLAVAMEV,=,31.808753,3.459742
...,...,...,...,...,...,...,...
12155,human,HLA-A*02:01,15,EISTNIRQAGVQYSR,>,178571.428600,12.092744
12156,human,HLA-A*02:01,15,IRQAGVQYSRADEEQ,>,178571.428600,12.092744
12157,human,HLA-A*02:01,17,QFLKFSLPFPFLYKFLL,=,20681.845230,9.937012
12158,human,HLA-A*02:01,18,PLMGGAYIAFPTSCHMFI,=,13161.358710,9.485040


In [8]:
# Check for duplicate sequences.
print(pep_data_filtered['sequence'].duplicated().sum())

0


In [9]:
# Inspect the minimum and maximum sequence lengths.
max_length = pep_data_filtered['sequence'].apply(lambda x: len(x)).max()
min_length = pep_data_filtered['sequence'].apply(lambda x: len(x)).min()
print(f"Maximum sequence length: {max_length}, Minimum sequence length: {min_length}")

Maximum sequence length: 30, Minimum sequence length: 8


In [10]:
# Select and rename the sequence and log-transformed score columns.
pep_data_filtered_clean = pep_data_filtered[['sequence', 'log_meas']].rename(columns={'sequence': 'seq', 'log_meas': 'score'})

In [11]:
pep_data_filtered_clean

,seq,score
0,ILGFVFTL,0.203286
1,FLGRIWPS,0.693147
2,FLGKIWPS,1.386294
3,FLGKIWSS,2.079442
4,GLAVAMEV,3.459742
...,...,...
12155,EISTNIRQAGVQYSR,12.092744
12156,IRQAGVQYSRADEEQ,12.092744
12157,QFLKFSLPFPFLYKFLL,9.937012
12158,PLMGGAYIAFPTSCHMFI,9.485040


## Random data split

### PARROT split file

In [12]:
# Shuffle the peptide data.
pep_data_filtered_clean_shuffled = pep_data_filtered_clean.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the shuffled data into training, validation, and test sets at a ratio of 0.8:0.1:0.1.
train_size = int(0.8 * len(pep_data_filtered_clean_shuffled))
dev_size = int(0.1 * len(pep_data_filtered_clean_shuffled))
pep_train = pep_data_filtered_clean_shuffled[:train_size]
pep_dev = pep_data_filtered_clean_shuffled[train_size:train_size + dev_size]
pep_test = pep_data_filtered_clean_shuffled[train_size + dev_size:]

In [13]:
pep_data_filtered_clean_shuffled

,seq,score
0,AASKQQMLM,11.266065
1,FQLNDTIHL,7.326531
2,ARAALQGGG,11.148282
3,VFTSRIQVI,9.903488
4,TVQEFIFSAL,5.771094
...,...,...
12155,TSTVEEQIQW,11.266065
12156,NLTYVQLES,9.301360
12157,ITTMDGDKL,9.614060
12158,GLQGIYVLV,1.098612


In [ ]:
# Save the shuffled peptide data.
pep_data_filtered_clean_shuffled.to_csv(plm_data + "parrot\\peptide\\peptide_random.txt", sep=" ", columns=["seq", "score"], header=False)

## Prepare PARROT optimization inputs

In [14]:
# Combine the training and validation data.
pep_train_dev = pd.concat([pep_train, pep_dev], ignore_index=True)
pep_train_dev

,seq,score
0,AASKQQMLM,11.266065
1,FQLNDTIHL,7.326531
2,ARAALQGGG,11.148282
3,VFTSRIQVI,9.903488
4,TVQEFIFSAL,5.771094
...,...,...
10939,KLIIDREVVA,9.666888
10940,SLQDNAVTV,2.772589
10941,FLYGGLLLA,0.095310
10942,TVANNPDDK,11.266065


In [15]:
# Check for duplicate sequences within each split.
print(pep_train_dev.shape[0], pep_test.shape[0])

# Check for sequences shared between splits.
print(set(pep_train_dev["seq"]) & set(pep_test["seq"]))

# Check the split sizes and proportions.
pep_train_dev.shape[0] / pep_data_filtered_clean_shuffled.shape[0], pep_test.shape[0] / pep_data_filtered_clean_shuffled.shape[0],

10944 1216
set()


(0.9, 0.1)

In [ ]:
# Save the combined training and validation data for PARROT optimization.
pep_train_dev.to_csv(plm_data + "parrot\\peptide\\peptide_train_dev.txt", sep=" ", columns=["seq", "score"], header=False)

## Save the split indices

In [16]:
# Collect the training, validation, and test indices in separate rows.
index_df = pd.DataFrame({
    "train": " ".join(map(str, pep_train.index)),
    "dev": " ".join(map(str, pep_dev.index)),
    "test": " ".join(map(str, pep_test.index))
}, index=[0])

# Transpose the index table.
index_df = index_df.T.reset_index()
index_df.columns = ["split", "indices"]

In [17]:
index_df

,split,indices
0,train,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18...
1,dev,9728 9729 9730 9731 9732 9733 9734 9735 9736 9...
2,test,10944 10945 10946 10947 10948 10949 10950 1095...


In [ ]:
# Write the split indices as unquoted text.
with open(plm_data + r"parrot\peptide\peptide_random_indices.txt", "w", encoding="utf-8") as f:
    for s in index_df["indices"]:
        f.write(str(s) + "\n")